In [1]:
# load the dataset cora
import torch
from torch_geometric.data import Data
from dataprocessing.partitioning import label_dirichlet_partition
from dataprocessing.datasets import GraphDataset
from dataprocessing.loaders import load_dataset, load_and_split, load_and_split_with_khop, load_and_split_with_feature_prop
import numpy as np


### Base test functions

In [2]:
def print_subgraph_stats(data: Data, name: str = "Subgraph"):
    """
    Prints basic statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object
        name: Name/identifier for the subgraph (default="Subgraph")
    """
    print(f"{name} stats:")
    print(f"Number of nodes: {data.num_nodes}")
    print(f"Number of edges: {data.edge_index.size(1)}")
    print(f"Number of features: {data.x.size(1)}")
    print(f"Number of classes: {len(torch.unique(data.y))}")
    print(f"Number of training nodes: {data.train_mask.sum().item()}")
    print(f"Number of validation nodes: {data.val_mask.sum().item()}")
    print(f"Number of test nodes: {data.test_mask.sum().item()}")
    print(f"Zero feature vectors: {(data.x.sum(dim=1) == 0).sum().item()}")
    print("-------------------")

# Example usage:
# print_subgraph_stats(subgraph, "Original subgraph")
# print_subgraph_stats(expanded_subgraph, "Expanded subgraph")

In [3]:
def check_connectivity_to_original(new_subgraph: Data, mapping: torch.Tensor):
    """
    Checks if all nodes in the new subgraph are connected to the original nodes.

    Args:
        new_subgraph: The expanded subgraph Data object.
        mapping: Tensor mapping original node indices to new subgraph indices.

    Returns:
        A boolean indicating if all nodes are connected to the original nodes.
    """
    # Set of original node indices in the new subgraph
    original_node_indices = set(mapping.tolist())

    # Check connectivity for each node in the new subgraph
    for node in range(new_subgraph.num_nodes):
        # Get neighbors of the current node
        mask = (new_subgraph.edge_index[0] == node) | (new_subgraph.edge_index[1] == node)
        neighbors = set(new_subgraph.edge_index[:, mask].unique().tolist())

        # Check if any neighbor is an original node
        if not neighbors.intersection(original_node_indices):
            print(f"Node {node} is not connected to any original node.")
            return False

    print("All nodes in the new subgraph are connected to the original nodes.")
    return True



In [4]:
from torch_geometric.utils import k_hop_subgraph
def get_subgraph_1_hop(subgraph: Data, full_graph: Data, node_indices: torch.Tensor):
    """
    Computes the 1-hop subgraph of nodes specified by indices within the full graph.
    
    Args:
        subgraph: Original subgraph Data object
        full_graph: The complete graph Data object
        node_indices: Tensor of node indices in the full graph to get 1-hop neighbors for
    
    Returns:
        Data object containing the 1-hop subgraph
    """
    # Create boolean mask for the specified nodes
    subgraph_nodes_mask = torch.zeros(full_graph.num_nodes, dtype=torch.bool)
    subgraph_nodes_mask[node_indices] = True
    subgraph_nodes = torch.where(subgraph_nodes_mask)[0]

    # Get the 1-hop subgraph
    subset, edge_index, mapping, edge_mask = k_hop_subgraph(
        subgraph_nodes, 
        1, 
        full_graph.edge_index, 
        relabel_nodes=True
    )

    # Extract features, labels, and masks from the full graph using the subset
    x = full_graph.x[subset]
    y = full_graph.y[subset]
    train_mask = full_graph.train_mask[subset]
    val_mask = full_graph.val_mask[subset]
    test_mask = full_graph.test_mask[subset]

    new_subgraph = Data(x=x, edge_index=edge_index, y=y, 
                       train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

    return new_subgraph, mapping

In [5]:
# def prepare_expanded_subgraph_for_propagation(original_subgraph: Data, expanded_subgraph: Data, original_indices: torch.Tensor):
#     """
#     Prepares expanded subgraph for feature propagation by:
#     - Zeroing features of new nodes (non-original nodes)
#     - Setting appropriate masks (only original nodes used for training)
#     - Maintaining original features and labels for initial nodes
#     """
#     # Determine device from original subgraph
#     device = original_subgraph.x.device
    
#     # Get the mapping of original nodes in the expanded graph
#     original_nodes_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
#     # The k_hop_subgraph function returns nodes in order where original nodes come first
#     # This is guaranteed by the relabel_nodes=True parameter
#     original_nodes_mask[:len(original_indices)] = True
    
#     # Print some verification info
#     print(f"Original nodes: {len(original_indices)}")
#     print(f"Expanded nodes: {expanded_subgraph.num_nodes}")
#     print(f"Original nodes in expanded graph: {original_nodes_mask.sum().item()}")
    
#     # Create new feature matrix (all zeros initially)
#     new_x = torch.zeros_like(expanded_subgraph.x, device=device)
#     # Copy original features for original nodes
#     new_x[original_nodes_mask] = original_subgraph.x
    
#     # Create new masks (only original nodes are used for training)
#     new_train_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
#     new_val_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
#     new_test_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
#     # Copy original masks for original nodes
#     new_train_mask[original_nodes_mask] = original_subgraph.train_mask
#     new_val_mask[original_nodes_mask] = original_subgraph.val_mask
#     new_test_mask[original_nodes_mask] = original_subgraph.test_mask
    
#     # Create new labels (zeros for new nodes)
#     new_y = torch.zeros(expanded_subgraph.num_nodes, dtype=expanded_subgraph.y.dtype, device=device)
#     new_y[original_nodes_mask] = original_subgraph.y
    
#     # Create new Data object
#     prepared_subgraph = Data(
#         x=new_x,
#         edge_index=expanded_subgraph.edge_index.to(device),  # Ensure edge_index is also on correct device
#         y=new_y,
#         train_mask=new_train_mask,
#         val_mask=new_val_mask,
#         test_mask=new_test_mask
#     )
    
#     return prepared_subgraph

In [6]:
def prepare_expanded_subgraph_for_propagation(original_subgraph: Data, expanded_subgraph: Data, mapping: torch.Tensor):
    """
    Prepares expanded subgraph for feature propagation by:
    - Zeroing features of new nodes (non-original nodes)
    - Setting appropriate masks based on original node mappings
    - Maintaining original features and labels for initial nodes
    """
    # Determine device from original subgraph
    device = original_subgraph.x.device
    
    # Create new feature matrix (all zeros initially)
    new_x = torch.zeros_like(expanded_subgraph.x, device=device)
    
    # Create new masks (all False initially)
    new_train_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_val_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    new_test_mask = torch.zeros(expanded_subgraph.num_nodes, dtype=torch.bool, device=device)
    
    # Copy original features for mapped nodes
    new_x[mapping] = original_subgraph.x
    
    # Get original train/val/test node indices
    orig_train_indices = torch.where(original_subgraph.train_mask)[0]
    orig_val_indices = torch.where(original_subgraph.val_mask)[0]
    orig_test_indices = torch.where(original_subgraph.test_mask)[0]
    
    # Map these indices to their new positions
    new_train_mask[mapping[orig_train_indices]] = True
    new_val_mask[mapping[orig_val_indices]] = True
    new_test_mask[mapping[orig_test_indices]] = True
    
    # Create new labels (zeros for new nodes)
    new_y = torch.zeros(expanded_subgraph.num_nodes, dtype=expanded_subgraph.y.dtype, device=device)
    new_y[mapping] = original_subgraph.y
    
    # Create new Data object
    prepared_subgraph = Data(
        x=new_x,
        edge_index=expanded_subgraph.edge_index.to(device),
        y=new_y,
        train_mask=new_train_mask,
        val_mask=new_val_mask,
        test_mask=new_test_mask
    )
    
    return prepared_subgraph

In [7]:
def analyze_graph_connectivity(data):
    """
    Analyzes and prints connectivity statistics for a PyTorch Geometric Data object.
    
    Args:
        data: PyTorch Geometric Data object containing edge_index and number of nodes
    """
    import torch
    import networkx as nx
    from torch_geometric.utils import degree, to_networkx
    
    # 1. Calculate node degrees
    node_degrees = degree(data.edge_index[0], num_nodes=data.num_nodes)
    
    # 2. Find isolated nodes
    isolated_nodes = torch.where(node_degrees == 0)[0]
    print("=== Isolated Nodes Analysis ===")
    print(f"Number of isolated nodes: {len(isolated_nodes)}")
    # print(f"Isolated nodes: {isolated_nodes.tolist()}")
    
    # 3. Get degree distribution
    # print("\n=== Degree Distribution ===")
    # degree_counts = torch.bincount(node_degrees.long())
    # for deg, count in enumerate(degree_counts):
    #     if count > 0:
    #         print(f"Nodes with degree {deg}: {count}")
    
    # 4. Basic connectivity statistics
    print("\n=== Connectivity Statistics ===")
    # print(f"Total nodes: {data.num_nodes}")
    # print(f"Total edges: {data.edge_index.shape[1]}")
    print(f"Average degree: {node_degrees.float().mean():.2f}")
    print(f"Maximum degree: {node_degrees.max()}")
    print(f"Minimum degree: {node_degrees.min()}")
    
    # 5. Connected components analysis
    G = to_networkx(data, to_undirected=True)
    connected_components = list(nx.connected_components(G))
    print("\n=== Connected Components Analysis ===")
    print(f"Number of connected components: {len(connected_components)}")
    print(f"Sizes of connected components: {[len(c) for c in connected_components]}")
    
    # return {
    #     'isolated_nodes': isolated_nodes,
    #     'node_degrees': node_degrees,
    #     'connected_components': connected_components
    # }



In [8]:
def print_node_indices(subgraph):
    """
    Print the indices of training, validation, and test nodes in the given subgraph.

    Args:
        subgraph: A graph or subgraph object that contains train_mask, val_mask, and test_mask attributes.
    """
    print(f"Training nodes in the subgraph: {subgraph.train_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of training nodes: {subgraph.train_mask.sum().item()}")
    print(f"Validation nodes in the subgraph: {subgraph.val_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of validation nodes: {subgraph.val_mask.sum().item()}")
    print(f"Test nodes in the subgraph: {subgraph.test_mask.nonzero(as_tuple=True)[0].tolist()}")
    print(f"Number of test nodes: {subgraph.test_mask.sum().item()}")

### Print dataset stats

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [10]:
# load data with split
data, dataset, clients_data, test_data = load_and_split("Cora",device= device, num_clients=10, beta=0.5)
# print stats
# pick client 1 and get stats
client_1 = clients_data[1]
# print stats
print_subgraph_stats(client_1, "Client 1")

# print connectivity stats
analyze_graph_connectivity(client_1)


Client 1 stats:
Number of nodes: 367
Number of edges: 548
Number of features: 1433
Number of classes: 4
Number of training nodes: 18
Number of validation nodes: 62
Number of test nodes: 151
Zero feature vectors: 0
-------------------
=== Isolated Nodes Analysis ===
Number of isolated nodes: 122

=== Connectivity Statistics ===
Average degree: 1.49
Maximum degree: 11.0
Minimum degree: 0.0

=== Connected Components Analysis ===
Number of connected components: 147
Sizes of connected components: [47, 1, 113, 5, 1, 2, 2, 13, 2, 4, 9, 1, 1, 1, 1, 1, 5, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 9, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1]


### Test with base functions

In [11]:

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
from dataprocessing.partitioning import label_dirichlet_partition
from dataprocessing.partitioning import create_subgraph

In [12]:


dataset_name = "Cora"
num_clients = 10
beta = 0.5

data, dataset = load_dataset(dataset_name, device)

labels = data.y.cpu().numpy()
N = len(labels)
K = len(np.unique(labels))

split_data_indexes = label_dirichlet_partition(labels, N, K, num_clients, beta)
    
    # Create test data
initial_subgraphs = [create_subgraph(data, indices, device) for indices in split_data_indexes]

subgraph = initial_subgraphs[0]
full_graph = data
node_indices = split_data_indexes[0]

print(f"Subgraph nodes: {subgraph}")
print(f"Full graph nodes: {full_graph}")
new_subgraph, mapping = get_subgraph_1_hop(subgraph, full_graph, node_indices)

if new_subgraph is not None:
    print(f"New subgraph nodes: {new_subgraph}")


Subgraph nodes: Data(x=[103, 1433], edge_index=[2, 54], y=[103], train_mask=[103], val_mask=[103], test_mask=[103])
Full graph nodes: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
New subgraph nodes: Data(x=[344, 1433], edge_index=[2, 1056], y=[344], train_mask=[344], val_mask=[344], test_mask=[344])


In [13]:
print(split_data_indexes[0])

[703, 2278, 950, 1370, 1619, 128, 1974, 765, 2102, 1713, 129, 1981, 458, 113, 2015, 1353, 2703, 1882, 2269, 662, 1360, 1979, 272, 925, 2577, 1184, 601, 737, 1368, 1879, 2200, 1137, 2246, 490, 117, 1958, 1650, 1978, 888, 779, 1083, 56, 1374, 709, 671, 123, 1338, 365, 1104, 1673, 1203, 2007, 2625, 1696, 2188, 2309, 142, 96, 1181, 213, 573, 47, 1826, 1222, 434, 846, 2513, 116, 1226, 441, 164, 1902, 2017, 903, 1179, 1953, 130, 517, 2414, 1630, 1707, 2405, 1415, 2100, 803, 2010, 344, 1167, 816, 2639, 645, 1872, 2201, 717, 74, 1580, 2384, 1324, 249, 2235, 1466, 233, 448]


In [19]:
# check the min & max node indices in the edge indices of subgraph and new_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in new_subgraph: {new_subgraph.edge_index.max()}")
print(f"Minimum node index in new_subgraph: {new_subgraph.edge_index.min()}")

# number of nodes in subgraph and new_subgraph
print(f"Number of nodes in subgraph: {subgraph.num_nodes}")
print(f"Number of nodes in new_subgraph: {new_subgraph.num_nodes}")



Maximum node index in subgraph: 96
Minimum node index in subgraph: 8
Maximum node index in new_subgraph: 343
Minimum node index in new_subgraph: 0
Number of nodes in subgraph: 103
Number of nodes in new_subgraph: 344


In [20]:
# check zero vectors in the subgraph & new_subgraph
zero_vectors = (subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in subgraph: {zero_vectors}")
zero_vectors = (new_subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in new_subgraph: {zero_vectors}")



Number of zero vectors in subgraph: 0
Number of zero vectors in new_subgraph: 0


In [21]:
prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, new_subgraph, mapping)

# check the min & max node indices in the edge indices of subgraph and prepared_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph: {prepared_subgraph.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph: {prepared_subgraph.edge_index.min()}")



Maximum node index in subgraph: 96
Minimum node index in subgraph: 8
Maximum node index in prepared_subgraph: 343
Minimum node index in prepared_subgraph: 0


In [22]:
len(mapping)

103

In [23]:
mapping

tensor([  2,   3,   5,   9,  13,  14,  15,  18,  20,  21,  22,  23,  28,  32,
         33,  34,  38,  42,  43,  54,  58,  61,  65,  71,  75,  82,  86,  92,
         95,  96, 100, 101, 102, 105, 109, 111, 114, 117, 121, 124, 126, 129,
        131, 142, 143, 149, 151, 152, 153, 154, 157, 159, 161, 172, 173, 174,
        177, 178, 179, 180, 186, 192, 207, 212, 214, 215, 218, 221, 223, 224,
        226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271,
        273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325,
        333, 337, 339, 341, 343])

In [24]:
print_node_indices(prepared_subgraph)

Training nodes in the subgraph: [2, 3, 5, 9, 13, 14, 15, 18, 20, 21, 22]
Number of training nodes: 11
Validation nodes in the subgraph: [23, 28, 32, 33, 34, 38, 42, 43, 54, 58, 61, 65, 71, 75, 82, 86]
Number of validation nodes: 16
Test nodes in the subgraph: [224, 226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271, 273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325, 333, 337, 339, 341, 343]
Number of test nodes: 34


In [25]:
# Usage example:
results = analyze_graph_connectivity(subgraph)

=== Isolated Nodes Analysis ===
Number of isolated nodes: 61

=== Connectivity Statistics ===
Average degree: 0.52
Maximum degree: 3.0
Minimum degree: 0.0

=== Connected Components Analysis ===
Number of connected components: 76
Sizes of connected components: [1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 2, 1, 1, 1, 2, 1, 1, 1, 7, 2, 3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 3, 3, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [26]:
# Usage example:
results = analyze_graph_connectivity(new_subgraph)

=== Isolated Nodes Analysis ===
Number of isolated nodes: 0

=== Connectivity Statistics ===
Average degree: 3.07
Maximum degree: 13.0
Minimum degree: 1.0

=== Connected Components Analysis ===
Number of connected components: 23
Sizes of connected components: [274, 5, 3, 2, 3, 5, 7, 4, 5, 2, 2, 4, 2, 2, 3, 3, 2, 3, 5, 2, 2, 2, 2]


In [27]:
# check nodes with zero feature vectors in the prepared_subgraph
zero_vectors = (prepared_subgraph.x.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in prepared_subgraph: {zero_vectors}")





Number of zero vectors in prepared_subgraph: 241


#### Load data from codebase

In [28]:
# load and split with khop
kh_data, kh_dataset, kh_clients_data, kh_test_data = load_and_split_with_khop("Cora",device, num_clients=10, beta=0.5, hop=1)

In [29]:
kh_subgraph0 = kh_clients_data[0]

In [30]:
kh_subgraph0

Data(x=[344, 1433], edge_index=[2, 1056], y=[344], train_mask=[344], val_mask=[344], test_mask=[344], original_nodes_mask=[344])

In [31]:
# get the matrix of kh_subgraph0
kh_subgraph0_mat = kh_subgraph0.x
# print the shape of kh_subgraph0_mat
print(f"Shape of kh_subgraph0_mat: {kh_subgraph0_mat.shape}")



Shape of kh_subgraph0_mat: torch.Size([344, 1433])


In [32]:
# compare the shape of kh_subgraph0_mat and subgraph.x and new_subgraph.x and prepared_subgraph.x
print(f"Shape of subgraph.x: {subgraph.x.shape}")
print(f"Shape of kh_subgraph0_mat: {kh_subgraph0_mat.shape}")
print(f"Shape of new_subgraph.x: {new_subgraph.x.shape}")
print(f"Shape of prepared_subgraph.x: {prepared_subgraph.x.shape}")





Shape of subgraph.x: torch.Size([103, 1433])
Shape of kh_subgraph0_mat: torch.Size([344, 1433])
Shape of new_subgraph.x: torch.Size([344, 1433])
Shape of prepared_subgraph.x: torch.Size([344, 1433])


In [33]:
# check maximum and minimum node indices in kh_subgraph0_mat fromthe edge indices
print(f"Maximum node index in kh_subgraph0_mat: {kh_subgraph0.edge_index.max()}")
print(f"Minimum node index in kh_subgraph0_mat: {kh_subgraph0.edge_index.min()}")


Maximum node index in kh_subgraph0_mat: 343
Minimum node index in kh_subgraph0_mat: 0


In [34]:
# check of prepared_subgraph.x is similar to kh_subgraph0_mat
are_similar = torch.allclose(prepared_subgraph.x, kh_subgraph0_mat)
print(f"Are prepared_subgraph.x and kh_subgraph0_mat similar? {are_similar}")

Are prepared_subgraph.x and kh_subgraph0_mat similar? False


In [35]:
# check if edge indices of kh_subgraph0 & prepared_subgraph are the same
print(f"Edge indices of kh_subgraph0: {kh_subgraph0.edge_index}")
print(f"Edge indices of prepared_subgraph: {prepared_subgraph.edge_index}")
# print their shapes
print(f"Shape of kh_subgraph0.edge_index: {kh_subgraph0.edge_index.shape}")
print(f"Shape of prepared_subgraph.edge_index: {prepared_subgraph.edge_index.shape}")

# check min & max node indices in kh_subgraph0 & prepared_subgraph edge indices
print(f"Minimum node index in kh_subgraph0.edge_index: {kh_subgraph0.edge_index.min()}")
print(f"Maximum node index in kh_subgraph0.edge_index: {kh_subgraph0.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph.edge_index: {prepared_subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph.edge_index: {prepared_subgraph.edge_index.max()}")





























Edge indices of kh_subgraph0: tensor([[310, 311,  11,  ..., 128, 218, 169],
        [  0,   0,   1,  ..., 341, 342, 343]])
Edge indices of prepared_subgraph: tensor([[310, 311,  11,  ..., 128, 218, 169],
        [  0,   0,   1,  ..., 341, 342, 343]])
Shape of kh_subgraph0.edge_index: torch.Size([2, 1056])
Shape of prepared_subgraph.edge_index: torch.Size([2, 1056])
Minimum node index in kh_subgraph0.edge_index: 0
Maximum node index in kh_subgraph0.edge_index: 343
Minimum node index in prepared_subgraph.edge_index: 0
Maximum node index in prepared_subgraph.edge_index: 343


In [36]:
# check the number of nodes for training test & validation in kh_subgraph0 & prepared_subgraph & subgraph
print("Total number of nodes in kh_subgraph0: ", kh_subgraph0.num_nodes)
print(f"Number of nodes in kh_subgraph0: {kh_subgraph0.num_nodes}")
print(f"Number of nodes in prepared_subgraph: {prepared_subgraph.num_nodes}")
print(f"Number of nodes in subgraph: {subgraph.num_nodes}")

# check the number of training nodes for kh_subgraph0 & prepared_subgraph & subgraph
print(f"Number of training nodes in kh_subgraph0: {kh_subgraph0.train_mask.sum()}")
print(f"Number of training nodes in prepared_subgraph: {prepared_subgraph.train_mask.sum()}")
print(f"Number of training nodes in subgraph: {subgraph.train_mask.sum()}")

# check the number of test nodes for kh_subgraph0 & prepared_subgraph & subgraph
print(f"Number of test nodes in kh_subgraph0: {kh_subgraph0.test_mask.sum()}")
print(f"Number of test nodes in prepared_subgraph: {prepared_subgraph.test_mask.sum()}")
print(f"Number of test nodes in subgraph: {subgraph.test_mask.sum()}")

Total number of nodes in kh_subgraph0:  344
Number of nodes in kh_subgraph0: 344
Number of nodes in prepared_subgraph: 344
Number of nodes in subgraph: 103
Number of training nodes in kh_subgraph0: 11
Number of training nodes in prepared_subgraph: 11
Number of training nodes in subgraph: 11
Number of test nodes in kh_subgraph0: 34
Number of test nodes in prepared_subgraph: 34
Number of test nodes in subgraph: 34


In [37]:
from tabulate import tabulate

def display_graph_stats(graph, name="Graph"):
    """
    Displays graph statistics in a formatted table.
    
    Args:
        graph: PyTorch Geometric Data object
        name: Name of the graph for display purposes
    """
    # Move tensors to CPU for safe computation
    edge_index_cpu = graph.edge_index.cpu()
    
    # Collect statistics
    stats = [
        [name, "Value"],
        ["Total Nodes", graph.num_nodes],
        ["Total Edges", edge_index_cpu.shape[1]],
        ["Maximum Node Index", edge_index_cpu.max().item()],
        ["Minimum Node Index", edge_index_cpu.min().item()],
        ["Training Nodes", graph.train_mask.sum().item()],
        ["Validation Nodes", graph.val_mask.sum().item()],
        ["Test Nodes", graph.test_mask.sum().item()]
    ]
    
    # Create and display table
    print(tabulate(stats, headers="firstrow", tablefmt="grid"))

# Example usage:
# display_graph_stats(kh_subgraph0, "KH Subgraph")
# display_graph_stats(prepared_subgraph, "Prepared Subgraph")
# display_graph_stats(subgraph, "Subgraph")

# To compare multiple graphs side by side:
def compare_graphs(*graphs, names=None):
    """
    Displays statistics for multiple graphs side by side in a table.
    
    Args:
        *graphs: Variable number of PyTorch Geometric Data objects
        names: List of names for the graphs (optional)
    """
    if names is None:
        names = [f"Graph {i+1}" for i in range(len(graphs))]
    
    # Prepare headers
    headers = ["Metric"] + names
    
    # Prepare rows
    metrics = [
        "Total Nodes",
        "Total Edges",
        "Max Node Index",
        "Min Node Index",
        "Training Nodes",
        "Validation Nodes",
        "Test Nodes"
    ]
    
    rows = []
    for metric in metrics:
        row = [metric]
        for graph in graphs:
            if metric == "Total Nodes":
                value = graph.num_nodes
            elif metric == "Total Edges":
                value = graph.edge_index.shape[1]
            elif metric == "Max Node Index":
                value = graph.edge_index.cpu().max().item()
            elif metric == "Min Node Index":
                value = graph.edge_index.cpu().min().item()
            elif metric == "Training Nodes":
                value = graph.train_mask.sum().item()
            elif metric == "Validation Nodes":
                value = graph.val_mask.sum().item()
            elif metric == "Test Nodes":
                value = graph.test_mask.sum().item()
            row.append(value)
        rows.append(row)
    
    print(tabulate(rows, headers=headers, tablefmt="grid"))

# Example usage:


In [38]:
compare_graphs(
    kh_subgraph0, 
    prepared_subgraph, 
    new_subgraph,
    subgraph, 
    names=["KH Subgraph", "Prepared Subgraph", "New Subgraph", "Subgraph"]
) 

+------------------+---------------+---------------------+----------------+------------+
| Metric           |   KH Subgraph |   Prepared Subgraph |   New Subgraph |   Subgraph |
+==================+===============+=====================+================+============+
| Total Nodes      |           344 |                 344 |            344 |        103 |
+------------------+---------------+---------------------+----------------+------------+
| Total Edges      |          1056 |                1056 |           1056 |         54 |
+------------------+---------------+---------------------+----------------+------------+
| Max Node Index   |           343 |                 343 |            343 |         96 |
+------------------+---------------+---------------------+----------------+------------+
| Min Node Index   |             0 |                   0 |              0 |          8 |
+------------------+---------------+---------------------+----------------+------------+
| Training Nodes   | 

### More stuff

In [39]:
#chek the maximum and minium node indices in the edge indices of subgraph and prepared_subgraph
print(f"Maximum node index in subgraph: {subgraph.edge_index.max()}")
print(f"Minimum node index in subgraph: {subgraph.edge_index.min()}")
print(f"Maximum node index in prepared_subgraph: {prepared_subgraph.edge_index.max()}")
print(f"Minimum node index in prepared_subgraph: {prepared_subgraph.edge_index.min()}")
# do the same for new_subgraph
print(f"Maximum node index in new_subgraph: {new_subgraph.edge_index.max()}")
print(f"Minimum node index in new_subgraph: {new_subgraph.edge_index.min()}")


Maximum node index in subgraph: 96
Minimum node index in subgraph: 8
Maximum node index in prepared_subgraph: 343
Minimum node index in prepared_subgraph: 0
Maximum node index in new_subgraph: 343
Minimum node index in new_subgraph: 0


In [40]:
# # Example usage with verification
# expanded_subgraph = get_subgraph_1_hop(subgraph, full_graph, original_indices)
# prepared_subgraph = prepare_expanded_subgraph_for_propagation(subgraph, expanded_subgraph, original_indices)

# Verify
print(f"Original features sum: {subgraph.x.sum()}")
print(f"New features sum: {prepared_subgraph.x.sum()}")  # Should be same as original
print(f"Training nodes: {prepared_subgraph.train_mask.sum()}")  # Should match original

Original features sum: 1972.0
New features sum: 1972.0
Training nodes: 11


In [41]:
# print statistics for original
print_subgraph_stats(subgraph, "Original subgraph")
print_subgraph_stats(prepared_subgraph, "Prepared subgraph")

Original subgraph stats:
Number of nodes: 103
Number of edges: 54
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 0
-------------------
Prepared subgraph stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------


In [42]:
# now lets do feature propagation on the prepared subgraph
from dataprocessing.data_utils import propagate_features

# get nonzero feature vector mask, which is the mapping of the original nodes
zero_vector_mask = (prepared_subgraph.x == 0).all(dim=1)

In [43]:
len(zero_vector_mask)
# number of zero vectors
print(zero_vector_mask.sum())


tensor(241)


In [44]:


non_zero_vector_mask = ~zero_vector_mask

propagated_subgraph = propagate_features(prepared_subgraph.x, prepared_subgraph.edge_index, non_zero_vector_mask)
print(propagated_subgraph)


TypeError: propagate_features() missing 1 required positional argument: 'device'

In [275]:
propagated_features = propagated_subgraph

In [ ]:
# 1. Check if they're exactly equal
are_equal = torch.equal(propagated_subgraph, prepared_subgraph.x)
print("Are tensors exactly equal?", are_equal)


In [45]:
# 2. Check the difference in values
difference = torch.abs(propagated_subgraph - prepared_subgraph.x)
print("Maximum difference:", difference.max().item())
print("Average difference:", difference.mean().item())


NameError: name 'propagated_subgraph' is not defined

In [ ]:
num_changed = (propagated_features != prepared_subgraph.x).sum().item()
print(f"Number of changed elements: {num_changed} out of {propagated_features.numel()}")


In [ ]:
print("Original features sum:", prepared_subgraph.x.sum().item())
print("Propagated features sum:", propagated_features.sum().item())


In [ ]:
#  5. Check if any zero vectors became non-zero
original_zero_vectors = (prepared_subgraph.x.sum(dim=1) == 0).sum().item()
propagated_zero_vectors = (propagated_features.sum(dim=1) == 0).sum().item()
print(f"Zero vectors before: {original_zero_vectors}")
print(f"Zero vectors after: {propagated_zero_vectors}")

In [281]:
# load dataset with feature propagation
data, dataset, clients_data, test_data = load_and_split_with_feature_prop("Cora", num_clients=10, beta=0.5, hop=1)

In [284]:
features2 = clients_data[1].x

In [ ]:
# coun zero vectors in features2
zero_vectors = (features2.sum(dim=1) == 0).sum().item()
print(f"Number of zero vectors in features2: {zero_vectors}")
# count non zero vectors in features2
non_zero_vectors = (features2.sum(dim=1) != 0).sum().item()
print(f"Number of non zero vectors in features2: {non_zero_vectors}")



In [ ]:
# check if features2 is similar to propagated_features
are_similar = torch.allclose(features2, propagated_features)
print(f"Are features2 and propagated_features similar? {are_similar}")

# check the difference between features2 and propagated_features
difference = torch.abs(features2 - propagated_features)
print(f"Difference between features2 and propagated_features: {difference.max().item()}")

### Test the functions 

In [ ]:
data, dataset, clients_data, test_data, split_data_indexes = load_and_split("Cora", num_clients=10, beta=0.5)
subgraph = clients_data[0]

# print statistics for original
print_subgraph_stats(subgraph, "Loand and split")


In [ ]:
test_data[0]
print_subgraph_stats(test_data[0], "Test data")

In [ ]:
# test load and split with khop
data, dataset, clients_data, test_data  = load_and_split_with_khop("Cora", num_clients=10, beta=0.5, hop=1)


print_subgraph_stats(clients_data[0], "Load and split with khop")
print_subgraph_stats(test_data[0], "Test data")

In [233]:
kh_data = clients_data[0]
kh_initial_data = test_data[0]

### Rough work

In [46]:
from dataprocessing.partitioning import create_k_hop_subgraph, prepare_expanded_subgraph_for_propagation, create_subgraph, create_k_hop_subgraph, create_subgraph

In [47]:
# load cora dataset
data, dataset = load_dataset("Cora", device)
# create a subgraph


In [49]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [59]:
# Get initial partition
split_data_indexes = label_dirichlet_partition(labels, N, K, num_clients, beta)
# Create test data
initial_subgraphs = [create_subgraph(data, indices, device) for indices in split_data_indexes]

In [78]:
# import k_hop_subgraph from torch_geometric
from torch_geometric.utils import k_hop_subgraph


In [89]:
subset, edge_index, mapping, edge_mask = k_hop_subgraph(split_data_indexes[0], 1, data.edge_index, relabel_nodes=True)

In [99]:
print(f"Subset: {subset}")

Subset: tensor([  20,   26,   47,   56,   70,   74,   88,   93,   95,   96,   97,   99,
         107,  113,  116,  117,  118,  120,  123,  125,  128,  129,  130,  142,
         144,  145,  161,  163,  164,  193,  210,  211,  213,  233,  249,  259,
         260,  264,  272,  273,  306,  321,  344,  365,  370,  371,  374,  382,
         389,  392,  410,  412,  415,  424,  434,  436,  437,  440,  441,  446,
         447,  448,  451,  454,  456,  458,  461,  472,  474,  478,  483,  490,
         505,  514,  516,  517,  525,  528,  540,  541,  544,  552,  573,  578,
         586,  597,  601,  612,  614,  621,  640,  643,  645,  647,  661,  662,
         671,  676,  701,  702,  703,  709,  717,  725,  735,  737,  739,  747,
         748,  765,  778,  779,  793,  801,  803,  807,  815,  816,  826,  830,
         844,  846,  850,  854,  888,  891,  903,  904,  917,  925,  932,  950,
         952,  971,  973,  994, 1021, 1041, 1042, 1043, 1070, 1076, 1083, 1104,
        1113, 1117, 1118, 1122, 

In [100]:
mapping

tensor([100, 313, 131, 179, 212,  20, 260, 109, 285, 224,  21, 265,  65,  13,
        271, 174, 343, 235, 310,  95, 177, 263,  38, 129, 337, 154,  86, 105,
        178, 233, 298, 149, 306,  71,  15, 256, 215, 262, 124, 111, 142,   3,
        180, 101,  96,  18, 173,  43, 143, 218, 157, 268, 339, 221, 295, 316,
         23,   9, 153,  32,  82,   2, 226, 159,  54, 121, 333,  14, 161,  58,
         28, 244, 273, 126, 152, 252,  22,  75, 325, 214, 223, 321, 186, 283,
        114, 269,  42, 151, 117, 341,  92, 232, 299, 102,   5, 207, 319, 172,
         34, 303, 192,  33,  61])

In [93]:
# number of nodes in the subset
print(f"Number of nodes in the subset: {subset.shape[0]}")
# number of nodes in the mapping
print(f"Number of nodes in the mapping: {mapping.shape[0]}")
# number of nodes in the edge_mask
print(f"Number of nodes in the edge_mask: {edge_mask.shape[0]}")

Number of nodes in the subset: 344
Number of nodes in the mapping: 103
Number of nodes in the edge_mask: 10556


In [115]:
x = data.x
y = data.y
train_mask = data.train_mask
val_mask = data.val_mask
test_mask = data.test_mask

# create a Data object for the subset
subset_data = Data(
    x=x[subset],
    y=y[subset],
    train_mask=train_mask[subset],
    val_mask=val_mask[subset],
    test_mask=test_mask[subset],
    edge_index=edge_index,
    mapping=mapping 
)

In [191]:
def reset_subgraph_features(subset_data: Data, mapping: torch.Tensor) -> Data:
    """
    Reset features of non-original nodes to zero while maintaining graph structure and masks.
    Only original nodes specified in the mapping will be used for train/val/test splits.
    
    Args:
        subset_data (Data): PyG Data object containing the subgraph
        mapping (torch.Tensor): Tensor containing indices of original nodes in the subgraph
    
    Returns:
        Data: New PyG Data object with reset features for non-original nodes
    """
    # Create mask for original nodes in the subgraph
    subset_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    subset_mask[mapping] = True
    
    # Initialize tensors with zeros, maintaining the original size
    reset_x = torch.zeros_like(subset_data.x)
    reset_y = torch.zeros_like(subset_data.y)
    
    # Copy only the original nodes' data, leaving others as zeros
    reset_x[subset_mask] = subset_data.x[subset_mask]
    reset_y[subset_mask] = subset_data.y[subset_mask]
    
    # Create new masks based on the mapping
    reset_train_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    reset_val_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    reset_test_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)

    # Set the masks for original nodes only
    for orig_idx in mapping:
        if subset_data.train_mask[orig_idx]:
            reset_train_mask[orig_idx] = True
        if subset_data.val_mask[orig_idx]:
            reset_val_mask[orig_idx] = True
        if subset_data.test_mask[orig_idx]:
            reset_test_mask[orig_idx] = True
    
    # Create new Data object
    reset_data = Data(
        x=reset_x,
        y=reset_y,
        train_mask=reset_train_mask,
        val_mask=reset_val_mask,
        test_mask=reset_test_mask,
        edge_index=subset_data.edge_index,
        mapping=mapping
    )
    
    return reset_data

reset_data = reset_subgraph_features(subset_data, mapping)
print_subgraph_stats(reset_data, "Reset subgraph features")






Reset subgraph features stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------


In [194]:
# check the labels of the nodes in the subgraph number of labels and the distribution of the labels
print(f"Number of labels in the subgraph: {reset_data.y.unique().shape[0]}")
print(f"Distribution of labels in the subgraph: {reset_data.y.bincount()}")
# print the number of nodes with each label
print(f"Number of nodes with label 0: {reset_data.y.eq(0).sum().item()}")
print(f"Number of nodes with label 1: {reset_data.y.eq(1).sum().item()}")
print(f"Number of nodes with label 2: {reset_data.y.eq(2).sum().item()}")
print(f"Number of nodes with label 3: {reset_data.y.eq(3).sum().item()}")
print(f"Number of nodes with label 4: {reset_data.y.eq(4).sum().item()}")
print(f"Number of nodes with label 5: {reset_data.y.eq(5).sum().item()}")



Number of labels in the subgraph: 5
Distribution of labels in the subgraph: tensor([263,   0,   1,  14,   0,  56,  10])
Number of nodes with label 0: 263
Number of nodes with label 1: 0
Number of nodes with label 2: 1
Number of nodes with label 3: 14
Number of nodes with label 4: 0
Number of nodes with label 5: 56


In [186]:
def reset_subgraph_features(subset_data: Data, mapping: torch.Tensor) -> Data:
    """
    Reset features of non-original nodes to zero while maintaining graph structure and masks.
    Only original nodes specified in the mapping will be used for train/val/test splits.
    
    Args:
        subset_data (Data): PyG Data object containing the subgraph
        mapping (torch.Tensor): Tensor containing indices of original nodes in the subgraph
    
    Returns:
        Data: New PyG Data object with reset features for non-original nodes
    """
    # Create mask for original nodes in the subgraph
    subset_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    subset_mask[mapping] = True
    
    # Initialize tensors with zeros, maintaining the original size
    reset_x = torch.zeros_like(subset_data.x)
    
    # Initialize reset_y with -1 for all nodes
    reset_y = torch.full((subset_data.num_nodes,), -1, dtype=subset_data.y.dtype)
    
    # Copy only the original nodes' data, leaving others as -1
    reset_x[subset_mask] = subset_data.x[subset_mask]
    reset_y[subset_mask] = subset_data.y[subset_mask]  # Copy original labels for original nodes
    
    # Create new masks based on the mapping
    reset_train_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    reset_val_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)
    reset_test_mask = torch.zeros(subset_data.num_nodes, dtype=torch.bool)

    # Set the masks for original nodes only
    for orig_idx in mapping:
        if subset_data.train_mask[orig_idx]:
            reset_train_mask[orig_idx] = True
        if subset_data.val_mask[orig_idx]:
            reset_val_mask[orig_idx] = True
        if subset_data.test_mask[orig_idx]:
            reset_test_mask[orig_idx] = True
    
    # Create new Data object
    reset_data = Data(
        x=reset_x,
        y=reset_y,  # Now contains original labels and -1 for others
        train_mask=reset_train_mask,
        val_mask=reset_val_mask,
        test_mask=reset_test_mask,
        edge_index=subset_data.edge_index,
        mapping=mapping
    )
    
    return reset_data

reset_data = reset_subgraph_features(subset_data, mapping)
print_subgraph_stats(reset_data, "Reset subgraph features")

Reset subgraph features stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 6
Number of training nodes: 11
Number of validation nodes: 16
Number of test nodes: 34
Zero feature vectors: 241
-------------------


In [187]:
# print unique labels
print(f"Unique labels: {torch.unique(reset_data.y)}")
# check how many nodes have each of the labels
print(f"Number of nodes with label 0: {reset_data.y.eq(0).sum().item()}")
print(f"Number of nodes with label 1: {reset_data.y.eq(1).sum().item()}")
print(f"Number of nodes with label 2: {reset_data.y.eq(2).sum().item()}")
print(f"Number of nodes with label 3: {reset_data.y.eq(3).sum().item()}")
print(f"Number of nodes with label 4: {reset_data.y.eq(4).sum().item()}")
print(f"Number of nodes with label 5: {reset_data.y.eq(5).sum().item()}")
print(f"Number of nodes with label 6: {reset_data.y.eq(6).sum().item()}")
print(f"Number of nodes with label -1: {reset_data.y.eq(-1).sum().item()}")



Unique labels: tensor([-1,  0,  2,  3,  5,  6])
Number of nodes with label 0: 22
Number of nodes with label 1: 0
Number of nodes with label 2: 1
Number of nodes with label 3: 14
Number of nodes with label 4: 0
Number of nodes with label 5: 56
Number of nodes with label 6: 10
Number of nodes with label -1: 241


In [188]:
def update_unlabeled_node_labels(data: Data) -> None:
    """
    Updates the labels of unlabeled nodes in the graph based on the majority voting of their neighbors.

    Args:
        data (Data): PyG Data object containing the graph with node labels.
    """
    # Find unlabeled nodes
    unlabeled_nodes = torch.where(data.y == -1)[0]
    print(f"Number of unlabeled nodes: {unlabeled_nodes.shape[0]}")

    for node in unlabeled_nodes:
        # Get neighbors
        mask = (data.edge_index[0] == node) | (data.edge_index[1] == node)
        neighbors = data.edge_index[:, mask].unique()

        # Flatten the neighbors tensor and convert to a list
        neighbor_indices = neighbors.flatten().unique().tolist()
        
        print(f"Node {node} has neighbors: {neighbor_indices}")

        # Get labels of the neighbors
        neighbor_labels = data.y[neighbor_indices]
        print(f"Neighbor labels: {neighbor_labels}")

        # Drop -1 labels
        valid_neighbor_labels = neighbor_labels[neighbor_labels != -1]
        print(f"Valid neighbor labels: {valid_neighbor_labels}")

        # Apply majority voting if there are valid labels
        if valid_neighbor_labels.numel() > 0:
            new_label = torch.mode(valid_neighbor_labels).values.item()
            print(f"New label: {new_label}")

            # Update the label of the node
            data.y[node] = new_label
            print(f"Updated label for node {node}: {data.y[node]}")
        else:
            print(f"No valid neighbor labels for node {node}. Label remains -1.")

# Example usage
update_unlabeled_node_labels(reset_data)

Number of unlabeled nodes: 241
Node 0 has neighbors: [0, 310, 311]
Neighbor labels: tensor([-1,  5, -1])
Valid neighbor labels: tensor([5])
New label: 5
Updated label for node 0: 5
Node 1 has neighbors: [1, 11, 18, 327]
Neighbor labels: tensor([-1, -1,  6, -1])
Valid neighbor labels: tensor([6])
New label: 6
Updated label for node 1: 6
Node 4 has neighbors: [4, 58, 294]
Neighbor labels: tensor([-1,  5, -1])
Valid neighbor labels: tensor([5])
New label: 5
Updated label for node 4: 5
Node 6 has neighbors: [6, 22, 52, 105, 116, 170, 224, 235, 269, 270, 271, 272, 273, 292]
Neighbor labels: tensor([-1,  5, -1,  5, -1, -1,  0,  5,  0, -1,  6, -1,  5, -1])
Valid neighbor labels: tensor([5, 5, 0, 5, 0, 6, 5])
New label: 5
Updated label for node 6: 5
Node 7 has neighbors: [7, 131, 197]
Neighbor labels: tensor([-1,  6, -1])
Valid neighbor labels: tensor([6])
New label: 6
Updated label for node 7: 6
Node 8 has neighbors: [8, 64, 207, 213, 293, 297, 298, 299]
Neighbor labels: tensor([-1, -1,  0, -

In [190]:
# print unique labels
print(f"Unique labels: {torch.unique(reset_data.y)}")
# print number of nodes with each label
print(f"Number of nodes with label 0: {reset_data.y.eq(0).sum().item()}")
print(f"Number of nodes with label 1: {reset_data.y.eq(1).sum().item()}")
print(f"Number of nodes with label 2: {reset_data.y.eq(2).sum().item()}")
print(f"Number of nodes with label 3: {reset_data.y.eq(3).sum().item()}")
print(f"Number of nodes with label 4: {reset_data.y.eq(4).sum().item()}")
print(f"Number of nodes with label 5: {reset_data.y.eq(5).sum().item()}")
print(f"Number of nodes with label 6: {reset_data.y.eq(6).sum().item()}")
print(f"Number of nodes with label -1: {reset_data.y.eq(-1).sum().item()}")


Unique labels: tensor([0, 2, 3, 5, 6])
Number of nodes with label 0: 83
Number of nodes with label 1: 0
Number of nodes with label 2: 2
Number of nodes with label 3: 41
Number of nodes with label 4: 0
Number of nodes with label 5: 180
Number of nodes with label 6: 38
Number of nodes with label -1: 0


In [185]:
# lets do lable propagation here:
# lets identify the unlabeled nodes
unlabeled_nodes = torch.where(reset_data.y == -1)[0]
print(f"Number of unlabeled nodes: {unlabeled_nodes.shape[0]}")
# foe each unlabeled nodes, lets find its neighbors
for node in unlabeled_nodes:
    # Get neighbors
    mask = (reset_data.edge_index[0] == node) | (reset_data.edge_index[1] == node)
    neighbors = reset_data.edge_index[:, mask].unique()

    # Flatten the neighbors tensor and convert to a list
    neighbor_indices = neighbors.flatten().unique().tolist()
    
    print(f"Node {node} has neighbors: {neighbor_indices}")
    # prin the lable sof the neighbors
    # print(f"Labels of the neighbors: {reset_data.y[neighbor_indices]}")
    neighbour_lables = reset_data.y[neighbor_indices]
    print(f"Neighbour lables: {neighbour_lables}")
    # drop the -1 lables
    neighbour_lables = neighbour_lables[neighbour_lables != -1]
    print(f"Neighbour lables: {neighbour_lables}")
    # apply majority voting
    new_label = torch.mode(neighbour_lables).values
    print(f"New label: {new_label}")
    # update the label of the node
    reset_data.y[node] = new_label
    print(f"Updated label: {reset_data.y[node]}")
        








Number of unlabeled nodes: 241
Node 0 has neighbors: [0, 310, 311]
Neighbour lables: tensor([-1,  5, -1])
Neighbour lables: tensor([5])
New label: 5
Updated label: 5
Node 1 has neighbors: [1, 11, 18, 327]
Neighbour lables: tensor([-1, -1,  6, -1])
Neighbour lables: tensor([6])
New label: 6
Updated label: 6
Node 4 has neighbors: [4, 58, 294]
Neighbour lables: tensor([-1,  5, -1])
Neighbour lables: tensor([5])
New label: 5
Updated label: 5
Node 6 has neighbors: [6, 22, 52, 105, 116, 170, 224, 235, 269, 270, 271, 272, 273, 292]
Neighbour lables: tensor([-1,  5, -1,  5, -1, -1,  0,  5,  0, -1,  6, -1,  5, -1])
Neighbour lables: tensor([5, 5, 0, 5, 0, 6, 5])
New label: 5
Updated label: 5
Node 7 has neighbors: [7, 131, 197]
Neighbour lables: tensor([-1,  6, -1])
Neighbour lables: tensor([6])
New label: 6
Updated label: 6
Node 8 has neighbors: [8, 64, 207, 213, 293, 297, 298, 299]
Neighbour lables: tensor([-1, -1,  0, -1, -1, -1,  0,  0])
Neighbour lables: tensor([0, 0, 0])
New label: 0
Updat

In [157]:
def label_propagation(data: Data, num_iterations: int = 20) -> torch.Tensor:
    """
    Propagates labels through the graph using a simple majority voting scheme.
    Only propagates from nodes with valid labels (not -1) and preserves original labels.
    Designed for undirected graphs.

    Args:
        data (Data): PyG Data object containing the graph.
        num_iterations (int): Number of iterations for label propagation.

    Returns:
        torch.Tensor: Predicted labels for all nodes in the graph.
    """
    # Initialize labels
    labels = data.y.clone()  # Copy the original labels
    num_nodes = data.num_nodes
    
    # Create a mask for nodes that had original labels (not -1)
    original_label_mask = (labels != -1)
    print(f"Initial labeled nodes: {original_label_mask.sum().item()}")
    
    for iteration in range(num_iterations):
        new_labels = labels.clone()  # Create a copy to update
        updates_made = 0
        
        # Iterate through unlabeled nodes only
        unlabeled_nodes = torch.where(labels == -1)[0]
        
        for node in unlabeled_nodes:
            # Get neighbors (for undirected graph, just need to check both rows of edge_index)
            mask = (data.edge_index[0] == node) | (data.edge_index[1] == node)
            neighbors = data.edge_index[:, mask].unique()
            
            if len(neighbors) > 0:
                # Get labels of neighbors that have valid labels (not -1)
                valid_neighbor_labels = labels[neighbors][labels[neighbors] != -1]
                
                # Only update if we have valid neighbors
                if len(valid_neighbor_labels) > 0:
                    # Majority voting among valid neighbors
                    label_counts = torch.bincount(valid_neighbor_labels.long())
                    new_labels[node] = label_counts.argmax().item()
                    updates_made += 1
        
        # Preserve original labels
        new_labels[original_label_mask] = labels[original_label_mask]
        
        # print(f"Iteration {iteration + 1}:")
        # print(f"Updates made: {updates_made}")
        # print(f"Unlabeled nodes remaining: {(new_labels == -1).sum().item()}")
        # print(f"Unique labels: {torch.unique(new_labels)}")
        # print("---")
        
        # Check if labels have changed
        # if torch.all(labels == new_labels):
        #     print(f"Converged after {iteration + 1} iterations")
        #     break
            
        labels = new_labels  # Update labels for the next iteration

    # Final statistics
    print("\nFinal Statistics:")
    print(f"Total nodes: {num_nodes}")
    print(f"Originally labeled nodes: {original_label_mask.sum().item()}")
    print(f"Nodes still unlabeled: {(labels == -1).sum().item()}")
    print(f"Unique labels in final result: {torch.unique(labels)}")
    
    return labels

In [158]:
# Apply label propagation
predicted_labels = label_propagation(reset_data)

# Update the reset_data.y with the new predicted labels
reset_data.y = predicted_labels

# Verify the changes
print("\nVerification:")
print("Number of unlabeled nodes (-1):", (reset_data.y == -1).sum().item())
print("Unique labels after propagation:", torch.unique(reset_data.y))

Initial labeled nodes: 301

Final Statistics:
Total nodes: 344
Originally labeled nodes: 301
Nodes still unlabeled: 43
Unique labels in final result: tensor([-1,  0,  3,  5])

Verification:
Number of unlabeled nodes (-1): 43
Unique labels after propagation: tensor([-1,  0,  3,  5])


In [159]:
print("Number of unlabeled nodes (-1):", (reset_data.y == -1).sum().item())
print("Unique labels after propagation:", torch.unique(reset_data.y))

Number of unlabeled nodes (-1): 43
Unique labels after propagation: tensor([-1,  0,  3,  5])


In [149]:
predicted_labels = label_propagation(reset_data)

Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 3
Neighbor labels: tensor([-1, -1, -1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 4
Neighbor labels: tensor([-1, -1, -1, -1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 9
Neighbor labels: tensor([-1, -1, -1, -1, -1, -1, -1, -1, -1])
Len of neighbors: 13
Neighbor labels: tensor([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 7
Neighbor labels: tensor([-1, -1, -1, -1, -1, -1, -1])
Len of neighbors: 1
Neighbor labels: tensor([-1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 3
Neighbor labels: tensor([-1, -1, -1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 4
Neighbor labels: tensor([-1, -1, -1, -1])
Len of neighbors: 2
Neighbor labels: tensor([-1, -1])
Len of neighbors: 9
Neighbor labels: tensor([-1, -1, -1, -1, -1, -1, -1, -1, 

In [135]:
print(predicted_labels)

tensor([-1, -1,  5, -1, -1, -1, -1, -1, -1,  6, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  3, -1, -1, -1,
        -1, -1, -1, -1, -1,  3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1,  5, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  5,  5, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1,  3, -1, -1, -1, -1, -1, 

In [136]:
# print unique classes in the dataset
print(f"Unique classes in the dataset: {dataset.y.unique()}")

Unique classes in the dataset: tensor([0, 1, 2, 3, 4, 5, 6])


In [122]:
print_node_indices(reset_data)

Training nodes in the subgraph: [2, 3, 5, 9, 13, 14, 15, 18, 20, 21, 22]
Number of training nodes: 11
Validation nodes in the subgraph: [23, 28, 32, 33, 34, 38, 42, 43, 54, 58, 61, 65, 71, 75, 82, 86]
Number of validation nodes: 16
Test nodes in the subgraph: [224, 226, 232, 233, 235, 244, 252, 256, 260, 262, 263, 265, 268, 269, 271, 273, 283, 285, 295, 298, 299, 303, 306, 310, 313, 316, 319, 321, 325, 333, 337, 339, 341, 343]
Number of test nodes: 34


In [116]:
# Create mask for original nodes in the subgraph
subset_mask = torch.zeros(len(subset), dtype=torch.bool)
subset_mask[mapping] = True

# Initialize tensors with zeros, maintaining the original size
reset_subset_x = torch.zeros_like(subset_data.x)  # Same size as original features
reset_subset_y = torch.zeros_like(subset_data.y)  # Same size as original labels

# Copy only the original nodes' data, leaving others as zeros
reset_subset_x[subset_mask] = subset_data.x[subset_mask]
reset_subset_y[subset_mask] = subset_data.y[subset_mask]

# For masks, maintain original train/val/test split
reset_subset_train_mask = subset_data.train_mask.clone()  # Clone original masks
reset_subset_val_mask = subset_data.val_mask.clone()
reset_subset_test_mask = subset_data.test_mask.clone()

# Create new Data object
reset_subset_data = Data(
    x=reset_subset_x,          # Features: zeros for non-original nodes
    y=reset_subset_y,          # Labels: zeros for non-original nodes
    train_mask=reset_subset_train_mask,  # Original train mask preserved
    val_mask=reset_subset_val_mask,      # Original val mask preserved
    test_mask=reset_subset_test_mask,    # Original test mask preserved
    edge_index=edge_index,
    mapping=mapping
)

In [117]:
print_node_indices(subset_data)


Training nodes in the subgraph: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Number of training nodes: 23
Validation nodes in the subgraph: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
Number of validation nodes: 67
Test nodes in the subgraph: [224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322

In [118]:
# print stats for reset_subset_data
print_subgraph_stats(subset_data, "Reset subset data")


Reset subset data stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 7
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Zero feature vectors: 0
-------------------


In [119]:
print_node_indices(reset_subset_data)

Training nodes in the subgraph: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
Number of training nodes: 23
Validation nodes in the subgraph: [23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
Number of validation nodes: 67
Test nodes in the subgraph: [224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322

In [120]:
print_subgraph_stats(reset_subset_data, "Reset subset data")

Reset subset data stats:
Number of nodes: 344
Number of edges: 1056
Number of features: 1433
Number of classes: 5
Number of training nodes: 23
Number of validation nodes: 67
Number of test nodes: 120
Zero feature vectors: 241
-------------------
